<!-- nav -->
[← 8 · Python Patterns](08_02_Python_Patterns.ipynb) · [Home](../../README.md) · [8.04 · TypeScript Types →](08_04_TypeScript_Types.ipynb)

# 8.03 · Python RegExp

Python's regular expressions are strings its `re` module reads, in a grammar of their own, the Library Reference's
"Regular Expression Syntax":

```text
re.fullmatch(r"(?P<word>[a-z]+)-(?P=word)", text)    a word, a hyphen, the same word again
```

`Python RegExp` is that grammar's dialect: its kinds are the constructs `re`'s syntax lists, named and shaped as
regexpp, ECMAScript's reference parser, names the ones ECMAScript has too (a `pattern` of `alternative`s, a
`capturinggroup`, a `quantifier`, a `characterclass`), then Python's own, its `conditional`, `atomicgroup`, inline
`flags` and `comment`
([CAP-281: Python's, ECMAScript's and C++'s regular expressions as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-281.md)).
Its checks are `re`'s parser's, with its messages.

In Python3, `r"(?P<word>[a-z]+)"` is a string, and stays one: no reader guesses that a string is a regular expression
([PRI-025: never guess the schema](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-025)). A
regular expression is written as one in its dialect's holder, `S.Python.RegExp.Expression`, by whoever knows it is one.
This page builds regular expressions from a single character up, in the builders' fluent calls, prints each as `re`
reads it, validates them, and reads the editions that have its kinds. Like the rest of Part I, it starts from
`dialects()`; `regexp` holds a spec in the dialect's holder, and `char` writes a character. Python also compiles each
text with `re` itself, and checks that its parser accepts what the dialect accepts and refuses what it refuses.

In [1]:
import re

from mbse.Expressions.Framework import Standards, Symbolics, Terms
from mbse.Expressions.Python import Python3, RegExp as PythonRegExp
from toolkit import dialects

store = dialects()
S = store.Expressions
REGEXP = PythonRegExp.Expressions.DIALECT
ToText = PythonRegExp.Text.ToText
char = lambda c: lambda u: u.character(lambda h: h.value(ord(c)))  # the character c, its value a code point


def regexp(spec):
    """The regular expression `spec` writes, held by its dialect's holder, written in a transaction of its own."""
    with store.transaction():
        return S.Python.RegExp.Expression(lambda e: e.root(spec))


def show(held):
    """A regular expression's text, as `re` reads it, and its kind."""
    print(f"{ToText(held):28}", Terms.kind_of(Terms.term(held))[0])
    re.compile(ToText(held))  # Python's own `re` reads it (Python's notebook only: `re` is Python's)

<a id="T08.03.1"></a>
## 08.03.1 · A character, a sequence, alternatives

The smallest regular expression is one character, `a`: a `character`, its `value` a code point, 97. Characters in a
row, `ab`, are an `alternative`, its `elements` in order; alternatives joined by `|`, `ab|c`, are a `pattern`, the
root `re` parses a whole regular expression to:

In [2]:
sequence = lambda u: u.alternative(lambda a: a.elements([char("a"), char("b")]))  # ab
for spec in (char("a"), sequence,
             lambda u: u.pattern(lambda p: p.alternatives([sequence, lambda a: a.alternative(lambda s: s.elements([char("c")]))]))):
    show(regexp(spec))
form = Terms.form(Terms.term(regexp(char("a"))))
print(form.kind, f"value: {form.attributes['value']}")

a                            character
ab                           alternative
ab|c                         pattern
character value: 97


<a id="T08.03.2"></a>
## 08.03.2 · Classes, sets and quantifiers

A **character class**, `[a-z_]`, matches one character among its elements: characters, and ranges, a
`characterclassrange` of two characters; `[^...]` negates it. A **character set** is a class `re` names: `.` any
character, `\d` a digit, `\s` a space, `\w` a word character, each a `characterset` of its `setKind`, `\D` and the
others negated. A **quantifier** repeats its `element` between its `min` and its `max`, none for no bound: `*`, `+`, `?`
and `{2,5}` are its shortest spellings, the printer's, then `?` where it is lazy and `+` where it is possessive (Python
3.11):

In [3]:
lowercase = lambda e: e.characterclassrange(lambda r: r.min(char("a")).max(char("z")))
specs = [
    lambda u: u.characterclass(lambda c: c.elements([lowercase, char("_")])),
    lambda u: u.characterclass(lambda c: c.negate(True).elements([lowercase])),
    lambda u: u.characterset(lambda s: s.setKind("any")),
    lambda u: u.characterset(lambda s: s.setKind("digit")),
    lambda u: u.characterset(lambda s: s.setKind("word").negate(True)),
    lambda u: u.quantifier(lambda q: q.min(0).element(char("a"))),
    lambda u: u.quantifier(lambda q: q.min(1).element(lambda e: e.characterset(lambda s: s.setKind("digit")))),
    lambda u: u.quantifier(lambda q: q.min(0).max(1).element(char("a"))),
    lambda u: u.quantifier(lambda q: q.min(2).max(5).greedy(False).element(char("a"))),
    lambda u: u.quantifier(lambda q: q.min(1).possessive(True).element(char("a"))),
]
for spec in specs:
    show(regexp(spec))

[a-z_]                       characterclass
[^a-z]                       characterclass
.                            characterset
\d                           characterset
\W                           characterset
a*                           quantifier
\d+                          quantifier
a?                           quantifier
a{2,5}?                      quantifier
a++                          quantifier


<a id="T08.03.3"></a>
## 08.03.3 · Groups, assertions and references

A **group** gathers alternatives: `(?:a|b)` a `group`, `(a)` a `capturinggroup`, numbered as it opens, and
`(?P<word>a)` one with a name. An **assertion** matches a place, not a character: `^`, `$`, `\b`, each an `assertion`
of its `assertionKind`, and the lookarounds `(?=a)` and `(?<!a)`, which hold alternatives. A **backreference** matches
what a group matched, `\1` by its number or `(?P=word)` by its name, both its `ref`. Python adds a `conditional`,
`(?(1)a|b)`, an `atomicgroup`, `(?>a+)`, inline `flags`, `(?i)` and `(?i-m:a)`, and a `comment`, `(?#note)`. The opening
example, whole:

In [4]:
word = lambda u: u.capturinggroup(lambda g: g.name("word").alternatives([lambda a: a.alternative(lambda s: s.elements([
    lambda e: e.quantifier(lambda q: q.min(1).element(lambda c: c.characterclass(lambda k: k.elements([lowercase]))))]))]))
repeated = regexp(lambda u: u.pattern(lambda p: p.alternatives([lambda a: a.alternative(lambda s: s.elements([
    word, char("-"), lambda e: e.backreference(lambda b: b.ref("word"))]))])))
show(repeated)
alternative = lambda *elements: lambda a: a.alternative(lambda s: s.elements(list(elements)))
specs = [
    lambda u: u.group(lambda g: g.alternatives([alternative(char("a")), alternative(char("b"))])),
    lambda u: u.alternative(lambda s: s.elements([
        lambda e: e.capturinggroup(lambda g: g.alternatives([alternative(char("a"))])),
        lambda e: e.backreference(lambda b: b.ref(1))])),
    lambda u: u.alternative(lambda s: s.elements([
        lambda e: e.assertion(lambda a: a.assertionKind("start")), char("a"),
        lambda e: e.assertion(lambda a: a.assertionKind("word")),
        lambda e: e.assertion(lambda a: a.assertionKind("lookahead").alternatives([alternative(char("b"))])),
        lambda e: e.assertion(lambda a: a.assertionKind("lookbehind").negate(True).alternatives([alternative(char("c"))])),
        lambda e: e.assertion(lambda a: a.assertionKind("end"))])),
    lambda u: u.alternative(lambda s: s.elements([
        lambda e: e.quantifier(lambda q: q.min(0).max(1).element(
            lambda c: c.capturinggroup(lambda g: g.alternatives([alternative(char("a"))])))),
        lambda e: e.conditional(lambda c: c.ref(1).yes(alternative(char("b"))).no(alternative(char("c"))))])),
    lambda u: u.atomicgroup(lambda g: g.alternatives([alternative(lambda e: e.quantifier(lambda q: q.min(1).element(char("a"))))])),
    lambda u: u.alternative(lambda s: s.elements([lambda e: e.flags(lambda f: f.add("i")), char("a"),
                                                  lambda e: e.comment(lambda c: c.text("note"))])),
    lambda u: u.flags(lambda f: f.add("i").remove("m").alternatives([alternative(char("a"))])),
]
for spec in specs:
    show(regexp(spec))

(?P<word>[a-z]+)-(?P=word)   pattern
(?:a|b)                      group
(a)\1                        alternative
^a\b(?=b)(?<!c)$             alternative
(a)?(?(1)b|c)                alternative
(?>a+)                       atomicgroup
(?i)a(?#note)                alternative
(?i-m:a)                     flags


<a id="T08.03.4"></a>
## 08.03.4 · A character's spelling, kept beside its value

A character has many spellings in `re`: `A`, `\x41`, `\101`, `\N{LATIN CAPITAL LETTER A}`. Its value is one code
point, 65; its `raw`, where a reader kept it, is its spelling as written, which the printer writes back, beside the
value it encodes ([PRI-023: domain representations](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-023)).
Without a `raw`, the printer writes its default spelling, escaping what is special where it stands: `.` outside a
class, `-` within one. Validation checks that a `raw` spells its value, as `re`'s parser reads it:

In [5]:
spelled = [lambda u, raw=raw: u.character(lambda c: c.raw(raw).value(65))
           for raw in ("A", "\\x41", "\\101", "\\N{LATIN CAPITAL LETTER A}")]
for spec in spelled:
    held = regexp(spec)
    print(f"{ToText(held):28}", f"value: {Terms.form(Terms.term(held)).attributes['value']}", "|", REGEXP.validate(held) or "valid")
show(regexp(lambda u: u.alternative(lambda s: s.elements([char("."), lambda e: e.characterclass(lambda c: c.elements([char(".")]))]))))
show(regexp(lambda u: u.characterclass(lambda c: c.elements([char("-"), char("a")]))))
wrong = regexp(lambda u: u.character(lambda c: c.raw("b").value(97)))
print(ToText(wrong), "|", *REGEXP.validate(wrong))
assert REGEXP.validate(wrong) == ["a character's raw 'b' does not spell U+0061"]

A                            value: 65 | valid
\x41                         value: 65 | valid
\101                         value: 65 | valid
\N{LATIN CAPITAL LETTER A}   value: 65 | valid
\.[.]                        alternative
[\-a]                        characterclass
b | a character's raw 'b' does not spell U+0061


<a id="T08.03.5"></a>
## 08.03.5 · Validated, as `re` validates it

Validation reports what `re`'s parser refuses, in its words: a quantifier's bounds out of order, a quantifier with
nothing to repeat, a group's name defined twice, a reference to a group the pattern does not have. A pattern's groups
and references are checked at its root, where they are numbered. Each refused term is kept, and printed as written
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [6]:
refused = [
    lambda u: u.quantifier(lambda q: q.min(5).max(2).element(char("a"))),
    lambda u: u.quantifier(lambda q: q.min(0).element(lambda a: a.assertion(lambda s: s.assertionKind("start")))),
    lambda u: u.pattern(lambda p: p.alternatives([alternative(
        lambda e: e.capturinggroup(lambda g: g.name("n").alternatives([alternative(char("a"))])),
        lambda e: e.capturinggroup(lambda g: g.name("n").alternatives([alternative(char("b"))])))])),
    lambda u: u.pattern(lambda p: p.alternatives([alternative(
        lambda e: e.capturinggroup(lambda g: g.alternatives([alternative(char("a"))])), lambda e: e.backreference(lambda b: b.ref(2)))])),
]
for spec in refused:
    held = regexp(spec)
    problems = REGEXP.validate(held)
    print(f"{ToText(held):16}", *problems)
    try:  # `re` refuses it too, with the same message (Python's notebook only: `re` is Python's)
        re.compile(ToText(held))
        raise AssertionError("compiled")
    except re.error as error:
        assert problems[0] == error.msg, (problems, error.msg)

a{5,2}           min repeat greater than max repeat
^*               nothing to repeat
(?P<n>a)(?P<n>b) redefinition of group name 'n' as group 2; was group 1
(a)\2            invalid group reference 2


<a id="T08.03.6"></a>
## 08.03.6 · Held by its holder alone, binding nothing, not evaluated

No Python3 expression holds a regular expression: Python3 has strings, and `re.compile(r"a+")` calls a function with
one, a Python3 `call` of a `literal` (tutorial 6), which nothing reads as anything but a string. So only its holder holds
one, while each dialect of the family reads the others' kinds and prints through one printer (tutorial 8, 08.5). A
named group's name is a label its backreferences resolve, not a name the expression binds: a regular expression needs
nothing from a scope. And it is matched by `re`, a library, not evaluated: Python3's evaluator refuses each kind by its
dialect's name:

In [7]:
variable = lambda name: lambda d: d.variable(lambda v: v.name(name))
with store.transaction():
    call = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
        .func(lambda f: f.attribute(lambda a: a.value(variable("re")).attr("compile")))
        .args([lambda a: a.literal(lambda r: r.value(lambda v: v.as_scalar("a+")).text('r"a+"').quote('"'))]))))
print(Python3.Text.ToText(call), "| its argument:", Terms.kind_of(Terms.form(Terms.term(call)).arguments[-1])[0],
      "of", Terms.dialect_of(Terms.form(Terms.term(call)).arguments[-1]))
print("Python RegExp reads", ", ".join(d.name() for d in REGEXP.reads()), "| one printer:", ToText is Python3.Text.ToText)
print(ToText(repeated), "| free:", ", ".join(Symbolics.free(repeated, REGEXP)) or "none")
try:
    Python3.Evaluators.OfAny(repeated)
    raise AssertionError("evaluated")
except NotImplementedError as error:
    print(error)

re.compile(r"a+") | its argument: literal of Expressions.Python.Python3.Term
Python RegExp reads Python3, Python Types, Python Patterns | one printer: True
(?P<word>[a-z]+)-(?P=word) | free: none
Python RegExp pattern is not evaluated


<a id="T08.03.7"></a>
## 08.03.7 · Its editions as data

`Python RegExp` covers Python 3.0 to 3.15, each found by its published name, and records which of its kinds each has
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
every one since 3.0 but `atomicgroup`, from 3.11. What an edition added within a kind, such as 3.11's possessive
quantifiers, is not recorded apart:

In [8]:
dialect = store.Instances.Expressions.Python.RegExp
editions = Standards.standards(dialect)
print(len(editions), "editions:", list(editions)[0], "to", list(editions)[-1])
for version in ("Python 3.10", "Python 3.11"):
    kinds = [kind.name.rsplit(".", 1)[1] for kind in Standards.kinds(Standards.find(dialect, version))]
    print(f"{version:11}", len(kinds), "kinds, AtomicGroup among them:", "AtomicGroup" in kinds)
atomic = store.registered("Expressions.Python.RegExp.AtomicGroup")
print("AtomicGroup since", next(iter(Standards.of(dialect, atomic))))

16 editions: Python 3.0 to Python 3.15
Python 3.10 14 kinds, AtomicGroup among them: False
Python 3.11 15 kinds, AtomicGroup among them: True
AtomicGroup since Python 3.11


## What you learned

- **A regular expression is a term of its own dialect**, `Python RegExp`, named as regexpp names what ECMAScript shares
  with it, Python's own constructs beside them; held by its holder, never guessed from a string.
- **It is built from a character up**: alternatives of elements, classes and sets, quantifiers, groups, assertions and
  references, and Python's conditionals, atomic groups, flags and comments.
- **A character keeps its spelling** beside its code point, and validation checks one spells the other.
- **Validation is `re`'s**, with its messages; a named group's name is a label, so it binds nothing.
- **It is not evaluated**: `re` matches it; Python3's evaluator refuses each kind by its dialect's name.
- **Its editions are data**: Python 3.0 to 3.15, `atomicgroup` from 3.11.

Next, the TypeScript family, starting with `TypeScript Types`.

<!-- nav -->
[← 8 · Python Patterns](08_02_Python_Patterns.ipynb) · [Home](../../README.md) · [8.04 · TypeScript Types →](08_04_TypeScript_Types.ipynb)